# Model Retention Frontier v1

## Governance

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` production composer.
- **Reference baseline**: `soxl_backtest_policy_reimplementation_v1`, the qlib-aligned SOXLBacktest buy/sell path.
- **Direct comparison baselines**: `model_retention_walkforward_gate_v1`, `cap10_wf2020_soxlbacktest_target_contract_v1`, `cap10_three_fold_bc_slice_screen_v1`, `cap10_three_fold_checkpoint_metric_replay_v2`, `state_objective_budget_usage_bc_official_replay_v1`.
- **Role**: `diagnose` and training handoff, not actor promotion.
- **Target window**: retention gate requires `wf_2020`, `wf_2022`, `wf_2023_2024`, plus YTD return alignment through the current safe replay window.
- **Changed layer**: `selection` / `model_selection` evidence routing.
- **External reference**: qlib-aligned SOXLBacktest buy/sell reference and existing cap10 target-contract screens.

## Hypothesis

The next useful training run should not start from another simple cap or renamed profile. The current frontier should point to the SOXLBacktest wf2020 target contract and three-fold BC/checkpoint path because existing retained actors fail either walk-forward return participation or YTD return alignment.

## Acceptance Criteria

1. Scan current model-selection summaries and retention gate thresholds.
2. Rank evidence by readiness for the next training run.
3. Keep SOXLBacktest as a reference baseline, not a promotable actor.
4. Write reproducible sidecars under this card.
5. End with a no-promotion decision unless a retained actor actually passes the gate.

## Closest Historical Cases

- **Closest success case**: `cap10_wf2020_soxlbacktest_target_contract_v1` created a non-leaking wf2020 target contract from the qlib-aligned SOXLBacktest replay.
- **Closest failure case**: `cap10_three_fold_checkpoint_metric_replay_v2` passed drawdown in all folds but failed wf2020 relative return by a large margin, so it cannot be retained.
- **Why this is not duplicate work**: this notebook does not rerun another target or checkpoint screen. It consolidates the current evidence into a ranked next-training frontier and prevents repeating failed low-participation/cap-only paths.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "soxl_quant").exists():
            return candidate
    raise RuntimeError("could not locate repo root containing src/soxl_quant")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.model_retention_frontier import (
    scan_model_retention_frontier,
    write_retention_frontier_outputs,
)

OUTPUT_DIR = ROOT / "runtime" / "model_selection" / "card" / "model_retention_frontier_v1"
result = scan_model_retention_frontier(root=ROOT)
write_retention_frontier_outputs(result, output_dir=OUTPUT_DIR)
result["summary"]

{'generated_at': '2026-06-13T17:59:19.358005+00:00',
 'rows_scanned': 92,
 'recommended_next_source': 'cap10_wf2020_soxlbacktest_target_contract_v1',
 'recommended_next_action': 'run_three_fold_bc_slice_screen_before_any_ppo',
 'recommended_primary_blocker': 'training_contract_ready',
 'required_best_return_pct': 213.143462,
 'reference_return_pct': 132.077978,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']}

In [2]:
frontier = pd.DataFrame(result["rows"])
frontier.head(12)[[
    "frontier_rank",
    "experiment",
    "primary_blocker",
    "failed_wf_windows",
    "worst_wf_max_drawdown_pct",
    "relative_return_gap_to_target_pp",
    "ytd_return_pct",
    "ytd_gap_to_best_pct",
    "next_action",
]]

,frontier_rank,experiment,primary_blocker,failed_wf_windows,worst_wf_max_drawdown_pct,relative_return_gap_to_target_pp,ytd_return_pct,ytd_gap_to_best_pct,next_action
0,1,cap10_wf2020_soxlbacktest_target_contract_v1,training_contract_ready,,NaN,NaN,NaN,NaN,run_three_fold_bc_slice_screen_before_any_ppo
1,2,cap10_three_fold_checkpoint_drift_selector_v1,walkforward_relative_return_gap,wf_2020,19.42,315.45,NaN,NaN,inspect_failed_fold_drawdown_or_reward_drift_b...
2,3,cap10_three_fold_checkpoint_metric_replay_v2,walkforward_relative_return_gap,wf_2020,18.80,322.75,NaN,NaN,inspect_failed_fold_drawdown_or_reward_drift_b...
3,4,state_objective_budget_usage_bc_official_repla...,joint_drawdown_return_gap,"wf_2020,wf_2023_2024",21.35,369.50,27.19,185.96,run_fold_local_ppo_only_if_ytd_and_wf_gap_is_n...
4,5,cap10_bc_sequence_risk_state_diagnostic_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,add_sequence_risk_state_cap_to_joint_contract_...
5,6,cap10_foldlocal_joint_target_contract_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,run_joint_contract_bc_screen_with_drawdown_dri...
6,7,cap10_foldlocal_multi_objective_readiness_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,build_fold_local_joint_sparse_release_drawdown...
7,8,cap10_joint_contract_bc_ppo_drift_screen_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,use_bc_as_checkpoint_reference_or_add_ppo_drif...
8,9,cap10_joint_contract_bc_slice_screen_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,run_walkforward_ppo_screen_with_bc_warmstart_a...
9,10,cap10_joint_contract_checkpoint_drift_selector_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,add_drawdown_drift_penalty_or_new_risk_state_b...


In [3]:
recommended = frontier.iloc[0].to_dict()
key_rows = frontier[frontier["experiment"].isin([
    "cap10_wf2020_soxlbacktest_target_contract_v1",
    "cap10_three_fold_checkpoint_metric_replay_v2",
    "state_objective_budget_usage_bc_official_replay_v1",
    "state_conditioned_overlay_gap_to_20_v1",
])][[
    "frontier_rank",
    "experiment",
    "primary_blocker",
    "failed_wf_windows",
    "wf_drawdown_gap_pct",
    "relative_return_gap_to_target_pp",
    "ytd_return_pct",
    "ytd_gap_to_soxlbacktest_pct",
    "ytd_gap_to_best_pct",
    "next_action",
]]
key_rows

,frontier_rank,experiment,primary_blocker,failed_wf_windows,wf_drawdown_gap_pct,relative_return_gap_to_target_pp,ytd_return_pct,ytd_gap_to_soxlbacktest_pct,ytd_gap_to_best_pct,next_action
0,1,cap10_wf2020_soxlbacktest_target_contract_v1,training_contract_ready,,NaN,NaN,NaN,NaN,NaN,run_three_fold_bc_slice_screen_before_any_ppo
2,3,cap10_three_fold_checkpoint_metric_replay_v2,walkforward_relative_return_gap,wf_2020,0.00,322.75,NaN,NaN,NaN,inspect_failed_fold_drawdown_or_reward_drift_b...
3,4,state_objective_budget_usage_bc_official_repla...,joint_drawdown_return_gap,"wf_2020,wf_2023_2024",1.35,369.50,27.19,104.89,185.96,run_fold_local_ppo_only_if_ytd_and_wf_gap_is_n...
70,71,state_conditioned_overlay_gap_to_20_v1,missing_walkforward_evidence,,NaN,NaN,NaN,NaN,NaN,


## Conclusion

- **Run result**: scanned the current model-selection card surface and wrote `retention_frontier.csv`, `summary.json`, and `conclusion.md`.
- **Decision**: no actor promotion and no actor retention from this notebook. The latest retention gate still has no promotable model.
- **Recommended next source**: `cap10_wf2020_soxlbacktest_target_contract_v1`.
- **Recommended next action**: `run_three_fold_bc_slice_screen_before_any_ppo`. This keeps the qlib-aligned SOXLBacktest buy/sell path as the baseline/teacher reference while requiring three-fold BC and checkpoint evidence before actor retention.
- **Key blocker**: the nearest checkpoint replay still fails `wf_2020` participation despite acceptable drawdown; the budget-usage BC official replay is close on drawdown but far below SOXLBacktest/current-best YTD return alignment.
- **Risk note**: do not repeat simple position caps or ad hoc overlays as the next training branch; existing evidence shows they trade off return alignment against drawdown instead of satisfying both.
- **Next action**: use this frontier as the entrypoint for the next actual retraining loop: merge the SOXLBacktest wf2020 target contract with the existing three-fold cap10 contract, run BC/checkpoint selection, then only retain an actor after full fixed-split, walk-forward, YTD, and dashboard replay gates pass.